# BEHAV3D Explorer — try it in your browser

This notebook runs the **real** BEHAV3D Explorer napari GUI on a free Google Colab machine and
streams the window to your browser. Nothing is installed on your computer.

<img src="https://raw.githubusercontent.com/imAIgene-Dream3D/BEHAV3D/main/docs/source/_static/screenshots/dock_widget_overview.png" width="700">

### How to use it

1. **Runtime ▸ Run all** (`Ctrl+F9` / `⌘+F9`)
2. Wait ~4 minutes while the environment and the demo dataset download
3. Click the link produced by the last step — the GUI opens in a **new browser tab**

*Optional, before you run:* **Runtime ▸ Change runtime type ▸ T4 GPU**. BEHAV3D runs fine on
CPU; a GPU only makes segmentation faster.

### What is in the demo

The demo dataset ships with **segmentation and tracking already computed**, so you can go
straight to the interesting parts: visualisation, track editing, feature extraction and the
behaviour analysis tabs.

### Good to know

* Colab gives you your own private machine — other visitors do not share your session.
* Colab disconnects after ~90 minutes of inactivity and ~12 hours maximum. **Nothing you do
  here is saved**; download anything you want to keep.
* The GUI renders with software OpenGL, so 2D views are smooth and the 3D view is slow.
  Stay in 2D for a comfortable tour.

---
## Step 1 · Fetch the BEHAV3D demo helper
Clones the repository and loads `demo/colab/colab_setup.py`, which contains everything below.
Takes a few seconds.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/imAIgene-Dream3D/BEHAV3D.git"
REPO_DIR = "/content/BEHAV3D"

if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

sys.path.insert(0, os.path.join(REPO_DIR, "demo", "colab"))
import colab_setup

print("helper loaded:", colab_setup.__file__)

---
## Step 2 · Build the demo (~4 minutes)

In order, this cell:

1. `apt-get`s the headless-Qt, OpenGL and VNC packages
2. downloads a **prebuilt** conda environment (solving it here would take 10+ minutes)
3. downloads the demo dataset and repoints its paths at this machine
4. starts a virtual display: `Xvfb → x11vnc → websockify → noVNC`
5. launches napari through BEHAV3D's own launcher (`launch_napari.py --internal`)

It is safe to re-run: anything already in place is skipped.

In [ ]:
colab_setup.bootstrap()

---
## Step 3 · Open the GUI

Runs the viewer in a **new browser tab**. If nothing happens, allow pop-ups for
`colab.research.google.com`, or run the Cloudflare fallback in the troubleshooting section.

In [ ]:
colab_setup.open_viewer()          # new tab
# colab_setup.open_viewer(in_tab=False)   # or embed it right here in the notebook

---
## What to do once the window opens

The BEHAV3D Explorer panel is docked on the right of the napari window.

1. **Data Preparation** tab → paste the two paths printed by Step 2:
   * Metadata CSV → `/content/behav3d_demo/metadata.csv`
   * Output folder → `/content/behav3d_demo/output`
   then click **Load**.
2. **Visualization** tab → pick a sample and display the raw channels next to the
   precomputed segmentation and tracks.
3. **Editing** tab → split, merge or delete a track and watch the layers update.
4. **Single cell / Analysis** tabs → the precomputed features are already there, so the
   plots and the results PDF render immediately.

Segmentation and tracking can be re-run from their tabs, but on a free CPU runtime that takes
a while — the demo ships their outputs precisely so you do not have to wait.

---
## Is this using a GPU?
BEHAV3D falls back to CPU everywhere, so a "No GPU" answer is fine.

In [ ]:
print(colab_setup.gpu_status())

---
## Troubleshooting

**The tab is blank / it never connects.** Colab's port proxy occasionally breaks WebSockets,
which noVNC needs. Run the Cloudflare cell below — it prints a plain `https://…trycloudflare.com`
link that works in any browser.

**Something failed during Step 2.** Run the health check, then read the log of whichever piece
is down (`napari`, `xvfb`, `x11vnc`, `novnc`, `cloudflared`).

**It was working and then froze.** The runtime probably disconnected. Re-run Step 2 — it
rebuilds only what is missing.

In [ ]:
colab_setup.status()

In [ ]:
print(colab_setup.tail("napari", 40))

In [ ]:
colab_setup.start_cloudflared()    # fallback route, no account needed

---
## Using your own data

Point the notebook at a bundle of your own — same layout as the demo
(`raw/`, `metadata.csv`, `output/`) — **before** running Step 2:

```python
import os
os.environ["BEHAV3D_DEMO_URL"] = "https://example.org/my_bundle.tar.gz"
```

Or upload files straight into `/content/behav3d_demo/` with the Files panel on the left, then
fix the recorded paths with:

```python
!/opt/behav3d/bin/python /content/BEHAV3D/demo/colab/prepare_demo.py --root /content/behav3d_demo
```

To install BEHAV3D properly on your own machine, follow the
[README](https://github.com/imAIgene-Dream3D/BEHAV3D#installation).